In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


# TLS/mTLS Credentials for Local Federated Learning

**Purpose.** Generate, inspect, and verify TLS credentials for local Flower SuperLink/SuperNode-style setups using the project's `generate_creds.py` script and OpenSSL. This is for **demo/local development certificates only**.

## Learning objectives
- Generate CA/server/client certificates in a temporary directory (no hardcoded paths).
- Verify certificate chains and demonstrate a negative verification test.
- Inspect certificate fields (subject, issuer, validity, SAN, key/signature info) via OpenSSL and optionally `cryptography`.
- Understand TLS 1.3/mTLS flows and PKI trust relationships at a high level.

## Background and math

### Public-key infrastructure (PKI)
A certificate binds an identity to a public key and is signed by a CA. Verification checks signatures, validity period, and name/SAN constraints.
*Meaning here:* the client/server must trust the CA that issued the peer's certificate for mTLS.

### mTLS
In mutual TLS, both server and client present certificates and verify each other.
*Meaning here:* local FL components can authenticate each other, reducing unauthorized connections (transport security, not DP).

## Assumptions and limitations
- Certificates generated here are **demo credentials** (short-lived, local SANs like localhost/127.0.0.1) and must not be used in production.
- Flower transport flags/configuration are environment-specific; this notebook validates credential material, not a live SuperGrid deployment.
- OpenSSL availability is required for generation/verification steps.


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    days_note: str = "script-defined"
cfg = Cfg(); cfg


In [ ]:
import os, json, shutil, subprocess, sys, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

openssl = shutil.which('openssl')
print('openssl:', openssl)
if openssl is None:
    print('[skip] openssl not found; TLS generation/verification cells will be skipped with messages.')


## Generate credentials in a temporary directory

In [ ]:
results = {'generated': False, 'verify_ok': None, 'negative_verify_failed_as_expected': None, 'cert_info': []}
tmp_obj = tempfile.TemporaryDirectory()
out = Path(tmp_obj.name) / 'tls_certs'
out.mkdir(parents=True, exist_ok=True)
print('temp cert dir:', out)
if openssl is None:
    print('[skip] cannot generate/verify without openssl')
else:
    cmd = [sys.executable, str(nb.ROOT / 'generate_creds.py'), '--output-dir', str(out)]
    print('running:', ' '.join(cmd))
    cp = subprocess.run(cmd, cwd=str(nb.ROOT), capture_output=True, text=True)
    print(cp.stdout)
    if cp.returncode != 0:
        print(cp.stderr)
        raise RuntimeError('generate_creds.py failed')
    print('generated files:', sorted([p.name for p in out.glob('*')]))
    results['generated'] = True


**How to read this / what it shows.** Credentials are created in a temporary directory and will be removed when the notebook finishes using it. No private key material is printed.

## Verify chain and negative test

In [ ]:
def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, p.stdout, p.stderr

if results['generated'] and openssl:
    ca = out/'ca.crt'; srv = out/'server.pem'; cli = out/'client.pem'
    rc, so, se = run([openssl,'verify','-CAfile',str(ca),str(srv)])
    print('server verify rc=', rc); print(so.strip() or se.strip())
    results['verify_ok'] = (rc == 0)
    # negative: verify server cert against a different CA (use client? no, create another CA quickly)
    bad = out/'badca'
    bad.mkdir(exist_ok=True)
    run([openssl,'genrsa','-out',str(bad/'ca.key'),'2048'])
    run([openssl,'req','-new','-x509','-days','1','-key',str(bad/'ca.key'),'-out',str(bad/'ca.crt'),'-subj','/CN=Bad-CA'])
    rc2, so2, se2 = run([openssl,'verify','-CAfile',str(bad/'ca.crt'),str(srv)])
    print('negative verify (wrong CA) rc=', rc2)
    print((so2+se2).strip())
    results['negative_verify_failed_as_expected'] = (rc2 != 0)
    assert results['verify_ok'], 'server cert should verify against its CA'
    assert results['negative_verify_failed_as_expected'], 'verification against wrong CA must fail'
    print('[ok] chain verification success and negative test failed as expected')
else:
    print('[skip] verification steps skipped')


**How to read this / what it shows.** A valid chain verifies against the issuing CA; the same server certificate must fail verification against an unrelated CA.

## Inspect certificates (OpenSSL and optional cryptography)

In [ ]:
def parse_cert_text(cert_path: Path):
    info = {'file': cert_path.name}
    if not openssl or not cert_path.exists():
        return info
    rc, so, se = run([openssl,'x509','-noout','-subject','-issuer','-dates','-serial','-ext','subjectAltName','-in',str(cert_path)])
    text = (so+'\n'+se).strip()
    info['openssl_text'] = text
    for line in text.splitlines():
        if '=' in line:
            k,v = line.split('=',1)
            info[k.strip().lower().replace(' ','_')] = v.strip()
        elif ':' in line and not line.strip().startswith('DNS:') and not line.strip().startswith('IP'):
            k,v = line.split(':',1)
            info.setdefault(k.strip().lower().replace(' ','_'), v.strip())
    return info

cert_table = []
if results['generated'] and openssl:
    for name in ['ca.crt','server.pem','client.pem']:
        p = out/name
        if p.exists():
            cert_table.append(parse_cert_text(p))
cert_df = pd.DataFrame(cert_table)
cert_df[[c for c in ['file','subject','issuer','notbefore','notafter','serial'] if c in cert_df.columns]] if not cert_df.empty else cert_df


In [ ]:
# Optional structured parsing with cryptography
crypto = nb.optional('cryptography')
crypto_rows = []
if crypto is not None and results['generated']:
    try:
        from cryptography import x509
        from cryptography.hazmat.primitives import hashes
        for name in ['ca.crt','server.pem','client.pem']:
            p = out/name
            if not p.exists(): continue
            cert = x509.load_pem_x509_certificate(p.read_bytes())
            san = None
            try:
                san = cert.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
                san_txt = ', '.join([str(x) for x in san])
            except Exception:
                san_txt = None
            crypto_rows.append({
                'file': name,
                'subject': cert.subject.rfc4514_string(),
                'issuer': cert.issuer.rfc4514_string(),
                'not_before': cert.not_valid_before_utc.isoformat() if hasattr(cert,'not_valid_before_utc') else str(cert.not_valid_before),
                'not_after': cert.not_valid_after_utc.isoformat() if hasattr(cert,'not_valid_after_utc') else str(cert.not_valid_after),
                'serial': hex(cert.serial_number),
                'sig_hash': cert.signature_hash_algorithm.name if cert.signature_hash_algorithm else None,
                'public_key_bits': getattr(cert.public_key(),'key_size',None),
                'san': san_txt,
            })
    except Exception as e:
        print('[cryptography parse skipped]', e)
crypto_df = pd.DataFrame(crypto_rows)
crypto_df if not crypto_df.empty else print('[note] cryptography structured table unavailable; OpenSSL parsing shown above')


**How to read this / what it shows.** Certificate metadata establishes identity, issuer, validity window, and SANs. Keys/signatures should be strong for real deployments; demo scripts prioritize convenience.

## Expiry countdown and hostname/SAN check

In [ ]:
def parse_not_after(cert_path: Path):
    if not openssl: return None
    rc, so, se = run([openssl,'x509','-noout','-enddate','-in',str(cert_path)])
    if rc!=0: return None
    # notAfter=Jun  1 12:00:00 2027 GMT (format varies)
    s = so.strip().split('=',1)[-1]
    for fmt in ['%b %d %H:%M:%S %Y %Z', '%b  %d %H:%M:%S %Y %Z']:
        try:
            return dt.datetime.strptime(s, fmt)
        except Exception:
            pass
    return None

expiry_rows=[]
if results['generated'] and openssl:
    now = dt.datetime.utcnow()
    for name in ['ca.crt','server.pem','client.pem']:
        p = out/name
        if not p.exists(): continue
        na = parse_not_after(p)
        days = (na-now).days if na else None
        expiry_rows.append({'file':name,'not_after_utc': str(na), 'days_remaining': days})
    # hostname/SAN presence check
    rc, so, se = run([openssl,'x509','-noout','-ext','subjectAltName','-in',str(out/'server.pem')])
    san_text = (so+se)
    print('server SAN extension:\n', san_text.strip() or '(none)')
    print('contains localhost/IP 127.0.0.1:', ('localhost' in san_text or '127.0.0.1' in san_text))
expiry_df = pd.DataFrame(expiry_rows)
expiry_df


## Diagrams and timeline

In [ ]:
# Mermaid flows (as markdown and simple figure)
mermaid_tls = r"""
```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    C->>S: ClientHello
    S->>C: ServerHello + Certificate + Finished
    C->>S: Finished
    C-->>S: Encrypted application data
```
"""
mermaid_mtls = r"""
```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    C->>S: ClientHello
    S->>C: ServerHello + Server Certificate
    S->>C: CertificateRequest
    C->>S: Client Certificate + Finished
    S->>C: Finished
```
"""
mermaid_pki = r"""
```mermaid
flowchart TB
    CA[Demo CA] -->|signs| SC[Server cert]
    CA -->|signs| CC[Client cert]
    SC -->|mTLS auth| CH[Secure channel]
    CC -->|mTLS auth| CH
```
"""
print(mermaid_mtls)


```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    C->>S: ClientHello
    S->>C: ServerHello + Server Certificate
    S->>C: CertificateRequest
    C->>S: Client Certificate + Finished
    S->>C: Finished
```

```mermaid
flowchart TB
    CA[Demo CA] -->|signs| SC[Server cert]
    CA -->|signs| CC[Client cert]
    SC -->|mTLS auth| CH[Secure channel]
    CC -->|mTLS auth| CH
```

**How to read this / what it shows.** mTLS requires both sides to present trusted certificates. PKI chain trust is rooted in the CA.


In [ ]:
# Figure 1: certificate validity timeline
import matplotlib.dates as mdates
fig, ax = plt.subplots(figsize=(8,3.5))
if not expiry_df.empty and openssl and results['generated']:
    labels=[]; starts=[]; ends=[]
    try:
        from datetime import timezone
        now = dt.datetime.now(timezone.utc).replace(tzinfo=None)
    except Exception:
        now = dt.datetime.utcnow()
    for name in ['ca.crt','server.pem','client.pem']:
        pth = out/name
        if not pth.exists(): continue
        rc1, s1, _ = run([openssl,'x509','-noout','-startdate','-in',str(pth)])
        rc2, s2, _ = run([openssl,'x509','-noout','-enddate','-in',str(pth)])
        def parse_line(s):
            s=s.strip().split('=',1)[-1]
            for fmt in ['%b %d %H:%M:%S %Y %Z','%b  %d %H:%M:%S %Y %Z']:
                try: return dt.datetime.strptime(s, fmt)
                except Exception: pass
            return None
        a=parse_line(s1); b=parse_line(s2)
        if a and b:
            labels.append(name); starts.append(a); ends.append(b)
    if labels:
        for i,(lab,a,b) in enumerate(zip(labels,starts,ends)):
            ax.barh(i, mdates.date2num(b)-mdates.date2num(a), left=mdates.date2num(a), height=0.5, color=['#2a6f97','#d1495b','#edae49'][i%3])
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.xaxis_date(); ax.set_title('Certificate validity timeline (demo certs)')
    else:
        ax.text(0.5,0.5,'Could not parse certificate dates', ha='center')
else:
    ax.text(0.5,0.5,'Timeline unavailable (openssl/certs missing)', ha='center')
fig.tight_layout(); nb.save_fig(fig, 'tls_validity_timeline'); plt.show()


**How to read this / what it shows.** Timelines and lifecycle diagrams highlight that certificates expire and must be rotated/revoked; demo certs use short, local-only validity.

## Expiry countdown figure


In [ ]:
# Figure 3: expiry countdown bar chart
fig, ax = plt.subplots(figsize=(6.5,3.5))
if not expiry_df.empty and 'days_remaining' in expiry_df.columns and expiry_df['days_remaining'].notna().any():
    vals = pd.to_numeric(expiry_df['days_remaining'], errors='coerce').fillna(0)
    ax.bar(expiry_df['file'], vals, color=['#2a6f97','#d1495b','#edae49'][:len(expiry_df)])
    ax.set_ylabel('Days remaining'); ax.set_title('Certificate expiry countdown (days)')
else:
    ax.text(0.5,0.5,'Expiry data unavailable', ha='center'); ax.set_title('Certificate expiry countdown')
fig.tight_layout(); nb.save_fig(fig, 'tls_expiry_countdown'); plt.show()


In [ ]:
# Figure: scatter of generated certificate remaining validity
if not expiry_df.empty and 'days_remaining' in expiry_df.columns:
    validity = pd.to_numeric(expiry_df['days_remaining'], errors='coerce').dropna().to_numpy()
else:
    validity = np.array([], dtype=float)
fig, ax = plt.subplots(figsize=(6.5,4))
if len(validity):
    ax.scatter(np.arange(len(validity)), validity, s=65, color='#00798c')
    ax.set_xticks(np.arange(len(validity))); ax.set_xticklabels(expiry_df.loc[pd.to_numeric(expiry_df['days_remaining'], errors='coerce').notna(), 'file'], rotation=20)
else:
    ax.text(0.5,0.5,'No generated certificate validity values',ha='center',va='center',transform=ax.transAxes)
ax.set_title('Generated certificate validity observations'); ax.set_xlabel('Certificate'); ax.set_ylabel('Days remaining')
fig.tight_layout(); nb.save_fig(fig, 'tls_validity_scatter'); plt.show()


In [ ]:
# Figure: histogram of generated certificate days remaining
fig, ax = plt.subplots(figsize=(6.5,4))
if len(validity):
    ax.hist(validity, bins=max(1,min(5,len(validity))), color='#edae49', edgecolor='white')
else:
    ax.text(0.5,0.5,'No generated certificate validity values',ha='center',va='center',transform=ax.transAxes)
ax.set_title('Distribution of generated certificate validity'); ax.set_xlabel('Days remaining'); ax.set_ylabel('Certificate count')
fig.tight_layout(); nb.save_fig(fig, 'tls_validity_histogram'); plt.show()


**How to read this / what it shows.** The histogram summarizes only the certificates created during this local notebook run. It is a small generated sample and does not describe an organization-wide certificate fleet.

## Quantitative summary


In [ ]:
summary = pd.DataFrame([results])
if not expiry_df.empty:
    display(expiry_df)
summary


## Sensitivity / ablation
- Key size, signature algorithm, and validity period affect security posture.
- SAN/hostname correctness is required for hostname verification in real clients.
- CA trust distribution determines who can participate in mTLS.

## Limitations and failure cases
- This notebook does not start Flower services or prove end-to-end secure transport in a live federation.
- Demo cert generation scripts may embed localhost-only assumptions.
- Private keys must be protected in real systems (never logged/committed).

## Conclusions and further experiments
- Demo TLS/mTLS materials can be generated and verified locally with OpenSSL and project tooling.
- Negative verification confirms chain trust boundaries.
- Further work: integrate with local SuperLink/SuperNode launch flags, automate rotation, and enforce stronger key/cipher policies.

## References
- E. Rescorla. "The Transport Layer Security (TLS) Protocol Version 1.3." RFC 8446, 2018.
- OpenSSL Project documentation (`openssl x509`, `openssl verify`).
- Flower documentation on TLS for local deployments (version-specific).


In [ ]:
# Cleanup temporary directory (no /tmp hardcoded path used)
try:
    tmp_obj.cleanup()
    print('temporary cert directory removed')
except Exception as e:
    print('cleanup note:', e)
